# 03. 모델링 & 앙상블
> 평가 지표: ROC-AUC

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# 전처리 결과 로드
with open('../data/processed.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
y_train = data['y_train']
X_test  = data['X_test']

print('X_train:', X_train.shape)
print('타겟 비율:', y_train.mean().round(4))

## 1. LightGBM 베이스라인

In [ ]:
SEED = 42
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1,
    'random_state': SEED,
    'n_jobs': -1
}

lgb_oof = np.zeros(len(X_train))
lgb_pred = np.zeros(len(X_test))
lgb_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = lgb.LGBMClassifier(**lgb_params, n_estimators=1000)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)]
    )

    oof_pred = model.predict_proba(X_val)[:, 1]
    lgb_oof[val_idx] = oof_pred
    lgb_pred += model.predict_proba(X_test)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, oof_pred)
    lgb_scores.append(score)
    print(f'Fold {fold+1} AUC: {score:.5f}')

lgb_cv = roc_auc_score(y_train, lgb_oof)
print(f'\nLightGBM CV AUC: {lgb_cv:.5f}')

## 2. XGBoost

In [ ]:
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': 0
}

xgb_oof = np.zeros(len(X_train))
xgb_pred = np.zeros(len(X_test))
xgb_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = xgb.XGBClassifier(**xgb_params, n_estimators=1000)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=50,
        verbose=False
    )

    oof_pred = model.predict_proba(X_val)[:, 1]
    xgb_oof[val_idx] = oof_pred
    xgb_pred += model.predict_proba(X_test)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, oof_pred)
    xgb_scores.append(score)
    print(f'Fold {fold+1} AUC: {score:.5f}')

xgb_cv = roc_auc_score(y_train, xgb_oof)
print(f'\nXGBoost CV AUC: {xgb_cv:.5f}')

## 3. CatBoost

In [ ]:
cat_oof = np.zeros(len(X_train))
cat_pred = np.zeros(len(X_test))
cat_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        eval_metric='AUC',
        random_seed=SEED,
        verbose=False,
        early_stopping_rounds=50
    )
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    oof_pred = model.predict_proba(X_val)[:, 1]
    cat_oof[val_idx] = oof_pred
    cat_pred += model.predict_proba(X_test)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, oof_pred)
    cat_scores.append(score)
    print(f'Fold {fold+1} AUC: {score:.5f}')

cat_cv = roc_auc_score(y_train, cat_oof)
print(f'\nCatBoost CV AUC: {cat_cv:.5f}')

## 4. 앙상블 (Weighted Average)

In [ ]:
# CV AUC 기반 가중치 자동 계산
scores = np.array([lgb_cv, xgb_cv, cat_cv])
weights = scores / scores.sum()
print(f'LGB: {weights[0]:.3f} | XGB: {weights[1]:.3f} | CAT: {weights[2]:.3f}')

# OOF 앙상블 AUC
ensemble_oof  = weights[0]*lgb_oof  + weights[1]*xgb_oof  + weights[2]*cat_oof
ensemble_pred = weights[0]*lgb_pred + weights[1]*xgb_pred + weights[2]*cat_pred

ensemble_cv = roc_auc_score(y_train, ensemble_oof)
print(f'\n앙상블 CV AUC: {ensemble_cv:.5f}')
print(f'\n모델별 요약')
print(f'  LightGBM : {lgb_cv:.5f}')
print(f'  XGBoost  : {xgb_cv:.5f}')
print(f'  CatBoost : {cat_cv:.5f}')
print(f'  앙상블   : {ensemble_cv:.5f}')

## 5. 제출 파일 생성

In [ ]:
import os
from datetime import datetime

submission = pd.read_csv('../data/sample_submission.csv')

# 타겟 컬럼명 확인 후 수정
pred_col = submission.columns[1]  # ID 다음 컬럼
submission[pred_col] = ensemble_pred

timestamp = datetime.now().strftime('%m%d_%H%M')
auc_str = f'{ensemble_cv:.5f}'.replace('.', 'p')
filename = f'../submissions/submission_{timestamp}_auc{auc_str}.csv'

os.makedirs('../submissions', exist_ok=True)
submission.to_csv(filename, index=False)
print(f'저장 완료: {filename}')
submission.head()

## 6. 피처 중요도 시각화

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 마지막 LightGBM 모델 기준 (fold 5)
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_  # 마지막 모델
}).sort_values('importance', ascending=False).head(30)

fig, ax = plt.subplots(figsize=(10, 10))
importance.plot(kind='barh', x='feature', y='importance', ax=ax, color='#2ecc71')
ax.set_title('Feature Importance TOP 30')
ax.invert_yaxis()
plt.tight_layout()
plt.show()